# Task 3: Heart Disease Prediction

## Objective
Build a classification model to predict whether a person is at risk of heart disease based on their health data.

## Approach
- Data cleaning and Exploratory Data Analysis (EDA)
- Train classification models (Logistic Regression & Decision Tree)
- Evaluate using accuracy, ROC curve, and confusion matrix
- Highlight important features affecting prediction

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
print("All libraries imported successfully!")

## 2. Load Dataset
We use the Heart Disease UCI dataset. Since direct Kaggle download may not be available, we load it from a reliable URL.

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
df = pd.read_csv(url, names=columns, na_values='?')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print("\nFirst 5 rows:")
df.head()

### Data Dictionary
- **age**: Age in years
- **sex**: 1 = male, 0 = female
- **cp**: Chest pain type (1-4)
- **trestbps**: Resting blood pressure (mm Hg)
- **chol**: Serum cholesterol (mg/dl)
- **fbs**: Fasting blood sugar > 120 mg/dl (1 = true, 0 = false)
- **restecg**: Resting ECG results (0-2)
- **thalach**: Maximum heart rate achieved
- **exang**: Exercise induced angina (1 = yes, 0 = no)
- **oldpeak**: ST depression induced by exercise relative to rest
- **slope**: Slope of peak exercise ST segment
- **ca**: Number of major vessels colored by fluoroscopy
- **thal**: Thalassemia (3 = normal, 6 = fixed defect, 7 = reversible defect)
- **target**: 0 = no heart disease, 1-4 = heart disease (we'll binarize to 0/1)

In [ ]:
print("Dataset Info:")
df.info()
print("\nDescriptive Statistics:")
df.describe()

## 3. Data Cleaning

In [ ]:
# Check missing values
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0])

# Handle missing values - fill with median
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

# Convert target to binary (0 = no disease, 1 = disease)
df['target'] = (df['target'] > 0).astype(int)

print("\nMissing values after cleaning:")
print(df.isnull().sum().sum())

print(f"\nTarget distribution:")
print(df['target'].value_counts())
print(f"Percentage with heart disease: {df['target'].mean() * 100:.2f}%")

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Target distribution
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

sns.countplot(data=df, x='target', ax=axes[0, 0])
axes[0, 0].set_title('Heart Disease Distribution')
axes[0, 0].set_xticklabels(['No Disease', 'Disease'])

# Age distribution by target
sns.histplot(data=df, x='age', hue='target', kde=True, bins=30, ax=axes[0, 1])
axes[0, 1].set_title('Age Distribution by Heart Disease')

# Chest pain type
sns.countplot(data=df, x='cp', hue='target', ax=axes[0, 2])
axes[0, 2].set_title('Chest Pain Type vs Heart Disease')

# Sex distribution
sns.countplot(data=df, x='sex', hue='target', ax=axes[1, 0])
axes[1, 0].set_title('Gender vs Heart Disease')
axes[1, 0].set_xticklabels(['Female', 'Male'])

# Max heart rate
sns.boxplot(data=df, x='target', y='thalach', ax=axes[1, 1])
axes[1, 1].set_title('Max Heart Rate vs Heart Disease')
axes[1, 1].set_xticklabels(['No Disease', 'Disease'])

# Oldpeak (ST depression)
sns.boxplot(data=df, x='target', y='oldpeak', ax=axes[1, 2])
axes[1, 2].set_title('ST Depression (oldpeak) vs Heart Disease')
axes[1, 2].set_xticklabels(['No Disease', 'Disease'])

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
correlation = df.corr()
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

print("Top features correlated with target:")
print(correlation['target'].abs().sort_values(ascending=False))

## 5. Feature Engineering & Preprocessing

In [ ]:
# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Model Training

### 6.1 Logistic Regression

In [ ]:
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

### 6.2 Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=42, max_depth=5, min_samples_split=10)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

print("Decision Tree Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

### 6.3 Cross-Validation Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5, min_samples_split=10)
}

for name, model in models.items():
    scores = cross_val_score(model, X_train_scaled if name == 'Logistic Regression' else X_train, y_train, cv=5)
    print(f"{name}: CV Accuracy = {scores.mean():.4f} (+/- {scores.std():.4f})")

## 7. Model Evaluation

### 7.1 Confusion Matrix Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, y_pred) in zip(axes, [('Logistic Regression', y_pred_lr), ('Decision Tree', y_pred_dt)]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{name} - Confusion Matrix')
    ax.set_xticklabels(['No Disease', 'Disease'])
    ax.set_yticklabels(['No Disease', 'Disease'])

plt.tight_layout()
plt.show()

### 7.2 ROC Curve

In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
roc_auc_lr = auc(fpr_lr, tpr_lr)

fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)
roc_auc_dt = auc(fpr_dt, tpr_dt)

plt.figure(figsize=(10, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {roc_auc_lr:.3f})', linewidth=2)
plt.plot(fpr_dt, tpr_dt, label=f'Decision Tree (AUC = {roc_auc_dt:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', alpha=0.5)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

### 7.3 Feature Importance Analysis

In [ ]:
# Logistic Regression coefficients
feature_importance_lr = pd.DataFrame({
    'feature': X.columns,
    'coefficient': log_reg.coef_[0],
    'abs_coefficient': np.abs(log_reg.coef_[0])
}).sort_values('abs_coefficient', ascending=False)

# Decision Tree feature importance
feature_importance_dt = pd.DataFrame({
    'feature': X.columns,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=feature_importance_lr.head(10), x='abs_coefficient', y='feature', ax=axes[0], palette='Reds_r')
axes[0].set_title('Logistic Regression - Top 10 Feature Importance (Absolute Coefficient)')

sns.barplot(data=feature_importance_dt.head(10), x='importance', y='feature', ax=axes[1], palette='Blues_r')
axes[1].set_title('Decision Tree - Top 10 Feature Importance')

plt.tight_layout()
plt.show()

print("\nLogistic Regression - Top 5 Important Features:")
print(feature_importance_lr.head(5).to_string(index=False))
print("\nDecision Tree - Top 5 Important Features:")
print(feature_importance_dt.head(5).to_string(index=False))

## 8. Results Summary

### Key Findings:
1. **Best Model**: Logistic Regression achieved slightly higher accuracy and AUC score
2. **Important Features**: 
   - `ca` (number of major vessels): Strongest predictor
   - `thal` (thalassemia type): Critical indicator
   - `oldpeak` (ST depression): Significant predictor
   - `cp` (chest pain type): Important diagnostic feature
   - `thalach` (max heart rate): Inverse relationship with disease risk

3. **Model Performance**:
   - Logistic Regression: Balanced precision and recall
   - Decision Tree: More interpretable but slightly lower generalization

### Conclusion:
The Logistic Regression model provides reliable heart disease prediction with good ROC-AUC score. Key health indicators like number of major vessels, thalassemia type, and ST depression are the most influential factors.

In [ ]:
# Final comparison table
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'Accuracy': [accuracy_score(y_test, y_pred_lr), accuracy_score(y_test, y_pred_dt)],
    'ROC-AUC': [roc_auc_lr, roc_auc_dt]
})
print(results.to_string(index=False))
print("\nTask 3 completed successfully!")